# 🐄🐃 AI-Based Cow & Buffalo Breed Classification Using Custom CNN
**Authors:** Amiya Ranjan Panda, Swaraj Kumar Behera, Prajakta Kuila, Vidya Mohanty, Subhashree Mishra, Manoj Kumar Mishra  
**Institution:** KIIT Deemed to be University, Bhubaneswar, India

---
This notebook implements the full pipeline described in the research paper:  
*Data Collection → Preprocessing → Augmentation → Model Training → Evaluation → Export*


## Phase 1 — Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Install dependencies
!pip install ImageHash --quiet


In [ ]:
# ── Unified Paths (change only here) ──────────────────────────────────────
RAW_DATASET       = "/content/drive/MyDrive/SIH/SIH_main"   # original nested dataset
FILTERED_DATASET  = "/content/drive/MyDrive/SIH/filtered"   # cleaned & split dataset
MODELS_DIR        = "/content/drive/MyDrive/SIH/models"     # saved models & artifacts

import os
os.makedirs(FILTERED_DATASET, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)
print("✅ Paths ready.")


## Phase 2 — Dataset Organization
Flatten nested `species/breed` folders into `species_breed` class folders,
then split images 80 / 10 / 10 into `train`, `valid`, and `test` sets.


In [ ]:
import os, shutil, pathlib, random

val_split  = 0.10   # 10% validation
test_split = 0.10   # 10% test

train_dir = os.path.join(FILTERED_DATASET, "train")
val_dir   = os.path.join(FILTERED_DATASET, "valid")
test_dir  = os.path.join(FILTERED_DATASET, "test")

for d in [train_dir, val_dir, test_dir]:
    os.makedirs(d, exist_ok=True)

dataset_path = pathlib.Path(RAW_DATASET)
print("🚀 Starting flattening and train/val/test split...")

for species_folder in dataset_path.iterdir():
    if not species_folder.is_dir():
        continue
    for breed_folder in species_folder.iterdir():
        if not breed_folder.is_dir():
            continue
        images = list(breed_folder.glob("*.jpg")) + list(breed_folder.glob("*.jpeg")) + list(breed_folder.glob("*.png"))
        if not images:
            continue

        folder_name = f"{species_folder.name}_{breed_folder.name}"
        random.shuffle(images)
        n = len(images)
        n_val   = max(1, int(n * val_split))
        n_test  = max(1, int(n * test_split))
        n_train = n - n_val - n_test

        splits = [(images[:n_train], train_dir),
                  (images[n_train:n_train+n_val], val_dir),
                  (images[n_train+n_val:], test_dir)]

        for img_list, dest_root in splits:
            dest = os.path.join(dest_root, folder_name)
            os.makedirs(dest, exist_ok=True)
            for img in img_list:
                shutil.copy(img, os.path.join(dest, img.name))

        print(f"🐾 {folder_name} | Total: {n} | Train: {n_train} | Val: {n_val} | Test: {n_test}")

print("\n✅ Dataset split complete!")


## Phase 3 — Data Preprocessing
Steps: duplicate removal → blur detection → corrupt image cleanup → resize to 128×128


In [ ]:
# Step 3a — Remove duplicate images (perceptual hash)
from PIL import Image
import imagehash

seen       = {}
duplicates = []

print("🔍 Scanning for duplicates...")
for root, _, files in os.walk(FILTERED_DATASET):
    for file in files:
        if file.lower().endswith((".jpg", ".jpeg", ".png")):
            path = os.path.join(root, file)
            try:
                h = imagehash.average_hash(Image.open(path))
                if h in seen:
                    duplicates.append(path)
                    os.remove(path)
                else:
                    seen[h] = path
            except Exception as e:
                print(f"⚠️  {path}: {e}")

print(f"✅ Duplicate scan done. Removed {len(duplicates)} duplicates.")


In [ ]:
# Step 3b — Remove blurry images (Laplacian variance via OpenCV)
import cv2

def is_blurry(path, threshold=50):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return True
    return cv2.Laplacian(img, cv2.CV_64F).var() < threshold

blurry = []
print("🔍 Scanning for blurry images...")
for root, _, files in os.walk(FILTERED_DATASET):
    for file in files:
        if file.lower().endswith((".jpg", ".jpeg", ".png")):
            path = os.path.join(root, file)
            try:
                if is_blurry(path):
                    blurry.append(path)
                    os.remove(path)
                    print(f"⚠️  Blurry removed: {path}")
            except Exception as e:
                print(f"Error: {path}: {e}")

print(f"✅ Blur detection done. Removed {len(blurry)} blurry images.")


In [ ]:
# Step 3c — Remove corrupt images & resize all to 128×128
removed_corrupt = 0
for split in ["train", "valid", "test"]:
    split_path = os.path.join(FILTERED_DATASET, split)
    for root, _, files in os.walk(split_path):
        for file in files:
            if file.lower().endswith((".jpg", ".jpeg", ".png")):
                path = os.path.join(root, file)
                img = cv2.imread(path)
                if img is None:
                    os.remove(path)
                    removed_corrupt += 1
                    continue
                img_resized = cv2.resize(img, (128, 128))
                cv2.imwrite(path, img_resized)

print(f"✅ Corrupt images removed: {removed_corrupt}. All remaining images resized to 128×128.")


In [ ]:
# Step 3d — Remove train/test cross-set duplicates (prevent data leakage)
def get_hashes(folder):
    h_map = {}
    for root, _, files in os.walk(folder):
        for f in files:
            if f.lower().endswith((".jpg", ".jpeg", ".png")):
                p = os.path.join(root, f)
                try:
                    h = str(imagehash.average_hash(Image.open(p)))
                    h_map.setdefault(h, []).append(p)
                except:
                    pass
    return h_map

train_hashes = get_hashes(train_dir)
test_hashes  = get_hashes(test_dir)

cross_dups = [(t, tr) for h, tlist in test_hashes.items() if h in train_hashes
              for t in tlist for tr in train_hashes[h]]

for dup_test, _ in cross_dups:
    try:
        os.remove(dup_test)
    except:
        pass

print(f"✅ Cross-set dedup done. Removed {len(cross_dups)} test images that appeared in train.")


## Phase 4 — Feature Engineering & Data Augmentation
Augmentation effectively triples the dataset size and improves generalization.


In [ ]:
import tensorflow as tf
import pathlib

# Load datasets
image_size = (128, 128)
batch_size = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir, labels="inferred", label_mode="categorical",
    image_size=image_size, batch_size=batch_size, shuffle=True
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    val_dir, labels="inferred", label_mode="categorical",
    image_size=image_size, batch_size=batch_size
)
test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir, labels="inferred", label_mode="categorical",
    image_size=image_size, batch_size=batch_size, shuffle=False
)

class_names  = train_ds.class_names
num_classes  = len(class_names)
print(f"Detected {num_classes} classes:", class_names)

# Augmentation layer (applied only to training)
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.15),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomTranslation(0.1, 0.1),
    tf.keras.layers.RandomBrightness(0.2),
    tf.keras.layers.RandomContrast(0.2),
])

rescale = tf.keras.layers.Rescaling(1./255)

AUTOTUNE = tf.data.AUTOTUNE

train_ds = (train_ds
            .map(lambda x, y: (data_augmentation(x, training=True), y))
            .map(lambda x, y: (rescale(x), y))
            .cache().shuffle(1000).prefetch(AUTOTUNE))

val_ds  = val_ds.map(lambda x, y: (rescale(x), y)).cache().prefetch(AUTOTUNE)
test_ds = test_ds.map(lambda x, y: (rescale(x), y)).cache().prefetch(AUTOTUNE)

print("✅ Datasets ready with augmentation.")


## Phase 5 — Custom CNN Model Training
5 convolutional blocks (32 → 64 → 128 → 256 → 512 filters) with dropout regularization,
trained for up to 100 epochs with early stopping and learning rate reduction.


In [ ]:
from tensorflow.keras import layers, models, callbacks

# ── Custom CNN Architecture ────────────────────────────────────────────────
def build_custom_cnn(num_classes):
    model = models.Sequential([
        # Block 1
        layers.Conv2D(32, (3,3), activation="relu", padding="same", input_shape=(128,128,3)),
        layers.MaxPooling2D(2,2),

        # Block 2
        layers.Conv2D(64, (3,3), activation="relu", padding="same"),
        layers.MaxPooling2D(2,2),

        # Block 3
        layers.Conv2D(128, (3,3), activation="relu", padding="same"),
        layers.MaxPooling2D(2,2),

        # Block 4
        layers.Conv2D(256, (3,3), activation="relu", padding="same"),
        layers.MaxPooling2D(2,2),

        # Block 5
        layers.Conv2D(512, (3,3), activation="relu", padding="same"),
        layers.MaxPooling2D(2,2),

        layers.Flatten(),
        layers.Dense(512, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation="softmax")
    ])
    return model

model = build_custom_cnn(num_classes)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
model.summary()


In [ ]:
# ── Callbacks ─────────────────────────────────────────────────────────────
cb_early_stop = callbacks.EarlyStopping(
    monitor="val_loss", patience=10, restore_best_weights=True, verbose=1
)
cb_reduce_lr = callbacks.ReduceLROnPlateau(
    monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6, verbose=1
)
cb_checkpoint = callbacks.ModelCheckpoint(
    filepath=os.path.join(MODELS_DIR, "best_model.keras"),
    monitor="val_accuracy", save_best_only=True, verbose=1
)

# ── Train ──────────────────────────────────────────────────────────────────
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=100,
    callbacks=[cb_early_stop, cb_reduce_lr, cb_checkpoint]
)


In [ ]:
# ── Training Curves ────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history["accuracy"],     label="Train Accuracy")
axes[0].plot(history.history["val_accuracy"],  label="Val Accuracy")
axes[0].set_title("Accuracy"); axes[0].legend(); axes[0].grid(True)

axes[1].plot(history.history["loss"],     label="Train Loss")
axes[1].plot(history.history["val_loss"],  label="Val Loss")
axes[1].set_title("Loss"); axes[1].legend(); axes[1].grid(True)

plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, "training_curves.png"), dpi=150)
plt.show()


## Phase 6 — Model Evaluation
Accuracy, Precision, Recall, F1-Score, Confusion Matrix, and Classification Report.


In [ ]:
import numpy as np
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

# Load best model for evaluation
best_model = tf.keras.models.load_model(os.path.join(MODELS_DIR, "best_model.keras"))

# Predict
y_pred_probs  = best_model.predict(test_ds)
y_pred        = np.argmax(y_pred_probs, axis=1)
y_true_onehot = np.concatenate([y for _, y in test_ds], axis=0)
y_true        = np.argmax(y_true_onehot, axis=1)

test_loss, test_acc = best_model.evaluate(test_ds, verbose=0)
print(f"\n🎯 Test Accuracy : {test_acc*100:.2f}%")
print(f"📉 Test Loss     : {test_loss:.4f}")


In [ ]:
# ── Confusion Matrix ───────────────────────────────────────────────────────
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(20, 16))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted"); plt.ylabel("True")
plt.title("Breed-Level Confusion Matrix")
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, "confusion_matrix.png"), dpi=150)
plt.show()


In [ ]:
# ── Classification Report ─────────────────────────────────────────────────
print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))


## Phase 7 — Model Export
Save the trained model in `.keras` format, export to `SavedModel` for Streamlit deployment,
and generate `model.json` with class name mappings.


In [ ]:
# Save .keras model
best_model.save(os.path.join(MODELS_DIR, "animal_classifier.keras"))
print("✅ Saved: animal_classifier.keras")


In [ ]:
# Export to TensorFlow SavedModel format (used by app.py via TFSMLayer)
best_model.export(os.path.join(MODELS_DIR, "animal_classifier_savedmodel"))
print("✅ Exported: animal_classifier_savedmodel/")


In [ ]:
# Generate model.json — class index to breed name mapping
import json

classes_dict = {str(i): name for i, name in enumerate(class_names)}
json_path = os.path.join(MODELS_DIR, "model.json")
with open(json_path, "w") as f:
    json.dump(classes_dict, f, indent=4)

print(f"✅ Saved model.json with {len(classes_dict)} classes.")
print(json.dumps(classes_dict, indent=2))


---
## ✅ Pipeline Complete

| Output | Path |
|--------|------|
| Best model (.keras) | `SIH/models/best_model.keras` |
| SavedModel (for app.py) | `SIH/models/animal_classifier_savedmodel/` |
| Class mapping | `SIH/models/model.json` |
| Confusion matrix | `SIH/models/confusion_matrix.png` |
| Training curves | `SIH/models/training_curves.png` |
